In [1]:
from model_ranking.dataclass import (
    ModelSourceConfig,
)
from typing import Dict, Any
from CCFV.utils.sliding_window_sampling import FeatureExtractor
from pytorch3dunet.unet3d.model import ResidualUNet2D

In [3]:
model_config = {
    "source_name": "EPFL",
    "model_name": "E_model_Res1",
    "model_type": "ResidualUNet2D",
}
model_cfg = ModelSourceConfig.model_validate(model_config)
model_cfg = model_cfg.create_config(feature_perturbation=None)

In [4]:
model = ResidualUNet2D(**model_cfg.model_dump())
print(model)

ResidualUNet2D(
  (encoders): ModuleList(
    (0): Encoder(
      (basic_module): ResNetBlock(
        (conv1): Conv2d(1, 64, kernel_size=(1, 1), stride=(1, 1))
        (conv2): SingleConv(
          (batchnorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (ReLU): ReLU(inplace=True)
        )
        (conv3): SingleConv(
          (batchnorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (non_linearity): ReLU(inplace=True)
      )
    )
    (1): Encoder(
      (pooling): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (basic_module): ResNetBlock(
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
        (conv2): SingleConv(
          (batchnorm): Bat

In [ ]:
config: Dict[str, Any] = {
    "target_datasets": [{"name": "EPFL"}],
    "source_models": [
        {"source_name": "EPFL", "model_name": "E_model_Unetr2", "model_type": "UnetrWrapper"}
    ],
    "source_model_base_path": (
        "/g/kreshuk/talks/segmentation_ModelSelection/experiments"
    ),
    "data_base_path": "/scratch/talks/data",
    "feature_cfg": {
        "layers": ["decoders.3"],  # Updated to correct layer name for ResidualUNet2D
        "sampling_seed": 42,
        "num_samples": 1000,
        #"output_dir_path": "/g/kreshuk/talks/model_ranking/notebooks/checks",
        "output_dir_path": None,  # Set to None for testing
    },
}

In [6]:
from model_ranking.dataclass import TransferFeatureExtractionConfig
from model_ranking.feature_ranking import TransferFeatureExtraction


feature_ranking_cfg = TransferFeatureExtractionConfig.model_validate(config)
feature_ranking = TransferFeatureExtraction(feature_ranking_cfg)

2025-08-07 09:16:56,113 [MainThread] INFO HDF5Dataset - Loading train set from: /scratch/talks/data/EPFL/test.h5...
2025-08-07 09:16:56,114 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
Global mean: 0.5599145293235779, global std: 0.11937469989061356
2025-08-07 09:16:56,599 [MainThread] INFO Dataset - Slice builder config: {'name': 'SliceBuilder', 'patch_shape': (1, 256, 256), 'stride_shape': (1, 256, 256), 'halo_shape': (0, 0, 0)}
2025-08-07 09:16:56,607 [MainThread] INFO HDF5Dataset - Number of patches: 750


In [7]:
target_dataset = feature_ranking.target_datasets["EPFL"]

In [8]:
feature_extractor = FeatureExtractor(model, layers=["decoders.3"])
for i, (image, label) in enumerate(iter(target_dataset)):
    print(f"Image {i}: shape={image.shape}, label={label.shape}")
    features = feature_extractor(image)
    print(f"Features from decoders.3: shape={features['decoders.3'].shape}")
    break

Image 0: shape=torch.Size([1, 1, 256, 256]), label=torch.Size([1, 1, 256, 256])
Features from decoders.3: shape=torch.Size([1, 64, 256, 256])
Features from decoders.3: shape=torch.Size([1, 64, 256, 256])


In [9]:
# Let's inspect the model structure to find the correct layer names
print("Available layers in the model:")
for name, module in model.named_modules():
    if name:  # Skip the root module
        print(f"  {name}: {type(module).__name__}")

# Focus on decoder layers specifically
print("\nDecoder layers:")
for name, module in model.named_modules():
    if "decoders" in name:
        print(f"  {name}: {type(module).__name__}")

Available layers in the model:
  encoders: ModuleList
  encoders.0: Encoder
  encoders.0.basic_module: ResNetBlock
  encoders.0.basic_module.conv1: Conv2d
  encoders.0.basic_module.conv2: SingleConv
  encoders.0.basic_module.conv2.batchnorm: BatchNorm2d
  encoders.0.basic_module.conv2.conv: Conv2d
  encoders.0.basic_module.conv2.ReLU: ReLU
  encoders.0.basic_module.conv3: SingleConv
  encoders.0.basic_module.conv3.batchnorm: BatchNorm2d
  encoders.0.basic_module.conv3.conv: Conv2d
  encoders.0.basic_module.non_linearity: ReLU
  encoders.1: Encoder
  encoders.1.pooling: MaxPool2d
  encoders.1.basic_module: ResNetBlock
  encoders.1.basic_module.conv1: Conv2d
  encoders.1.basic_module.conv2: SingleConv
  encoders.1.basic_module.conv2.batchnorm: BatchNorm2d
  encoders.1.basic_module.conv2.conv: Conv2d
  encoders.1.basic_module.conv2.ReLU: ReLU
  encoders.1.basic_module.conv3: SingleConv
  encoders.1.basic_module.conv3.batchnorm: BatchNorm2d
  encoders.1.basic_module.conv3.conv: Conv2d
  en

In [10]:
# Example: Extract features from different components of decoder 3
# You can target specific sub-modules within decoder 3:

# Options for decoder 3:
# "decoders.3" - entire decoder 3 output
# "decoders.3.basic_module" - output after the ResNetBlock
# "decoders.3.upsampling" - output after upsampling
# "decoders.3.basic_module.conv3" - output after the final conv in the ResNetBlock

# Let's try extracting from the entire decoder 3 and its basic_module
feature_extractor_detailed = FeatureExtractor(model, layers=["decoders.3", "decoders.3.basic_module"])
for i, (image, label) in enumerate(iter(target_dataset)):
    features_detailed = feature_extractor_detailed(image)
    print(f"Decoder 3 full output shape: {features_detailed['decoders.3'].shape}")
    print(f"Decoder 3 basic_module output shape: {features_detailed['decoders.3.basic_module'].shape}")
    break
    
feature_extractor_detailed.remove_handler()

Decoder 3 full output shape: torch.Size([1, 64, 256, 256])
Decoder 3 basic_module output shape: torch.Size([1, 64, 256, 256])
